# Executed bean classification evidence
Real source images; supplied partitions, no external-farm validity claim.

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score

root = Path.cwd() if Path("README.md").exists() else Path.cwd().parent
metrics = json.loads((root / "reports/metrics.json").read_text())
metrics["selected_on_validation"], metrics["test_calibrated"]

('mobilenet_transfer',
 {'accuracy': 0.875,
  'macro_f1': 0.8774744711535352,
  'log_loss': 0.3174269497394562,
  'multiclass_brier': 0.17577086565445674,
  'ece_10_bins': 0.049133530352264684})

In [2]:
pred = pd.read_csv(root / "reports/test-predictions.csv")
score = f1_score(pred.label, pred.predicted_label, average="macro")
assert abs(score - metrics["test_calibrated"]["macro_f1"]) < 1e-12
pred.groupby("label").size(), score

(label
 0    43
 1    43
 2    42
 dtype: int64,
 0.8774744711535352)

In [3]:
audit = json.loads((root / "reports/data-audit.json").read_text())
assert not audit["exact_duplicates_removed"]
audit

{'retained_counts': {'test': 128, 'train': 1034, 'validation': 133},
 'exact_duplicates_removed': [],
 'cross_split_dhash_distance_le_5': [],
 'limitation': 'Plant, farm and photographer identifiers unavailable; hash screening does not prove independent plants'}

In [4]:
from plantvision.inference import predict

image = root / pred.path.iloc[0]
predict(image.read_bytes(), root / "models/selected.pt")

{'label': 'bean_rust',
 'probabilities': {'angular_leaf_spot': 0.031612880527973175,
  'bean_rust': 0.9583072662353516,
  'healthy': 0.010079772211611271},
 'scope': 'Three bean classes only; no out-of-domain or agronomic diagnosis validation'}

The model is frozen after validation selection. Blur/darkness is a test robustness check. Farm identity, domain generalization and unknown-class rejection are unmeasured.